In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

3.4.1


In [3]:
spark = (SparkSession.builder
    .appName("preprocess-crypto")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/nifi/crypto-prices")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate())

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 13:33:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/24 13:33:27 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

# Quality Test

In [36]:
df = spark.read.parquet("hdfs://namenode:8020/nifi/crypto-prices/crypto-quotes-20251106-214622.parquet")

In [7]:
def test_nulls(df):
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [8]:
test_nulls(df)

[Stage 1:>                                                          (0 + 1) / 1]

+-------------+----+---+----+--------------+---------+------+
|current_price|high|low|open|previous_close|timestamp|symbol|
+-------------+----+---+----+--------------+---------+------+
|            0|   0|  0|   0|             0|        0|     0|
+-------------+----+---+----+--------------+---------+------+



In [9]:
from pyspark.sql.types import *
expected_schema = StructType([
    StructField("current_price", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("open", DoubleType(), True),
    StructField("previous_close", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
    StructField("symbol", StringType(), True)
])
def test_datatypes(df, expected_schema):
    return df.schema == expected_schema

In [10]:
test_datatypes(df, expected_schema)

True

# Preprocessing całego folderu i zapis do Hive

In [4]:
def rename_and_transform(df):
    # Wyciągnięcie symbolu
    df = df.withColumn(
        "Symbol",
        F.split("symbol", ":")[1].substr(1, 3)
    )

    # Timestamp → Datetime
    df = df.withColumn(
        "Datetime",
        F.from_unixtime(F.col("timestamp")).cast("timestamp")
    )
    
    # Mapowanie: stara_nazwa → nowa_nazwa
    rename_map = {
        "current_price": "CurrentPrice",
        "open": "OpeningPrice",
        "low": "LowestDayPrice",
        "high": "HighestDayPrice",
        "previous_close": "PreviousClosingPrice"
    }

    # Zmiana nazw
    df = df.withColumnsRenamed(rename_map)
    
    df = df.withColumn("PartitionDate", F.to_date("Datetime"))
    df= df.filter(df.PartitionDate.isNotNull())
    
    return df.select(
        "Symbol",
        "CurrentPrice",
        "OpeningPrice",
        "LowestDayPrice",
        "HighestDayPrice",
        "PreviousClosingPrice",
        "Datetime",
        "PartitionDate"
    )


In [7]:
df = spark.read.parquet("hdfs://namenode:8020/nifi/crypto-prices/")
df = rename_and_transform(df)

In [12]:
print(df.count())
df.describe().show()
df.show(5)

Py4JJavaError: An error occurred while calling o83.count.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(Unknown Source)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(Unknown Source)
java.base/java.lang.reflect.Constructor.newInstance(Unknown Source)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
py4j.Clien

In [22]:
# df.write \
#     .option("header", "true") \
#     .mode("overwrite") \
#     .csv("hdfs://namenode:8020/ready_data/crypto-prices")

In [9]:
(df.write
  .mode("append")
  .format("hive")
  .partitionBy("PartitionDate")
  .saveAsTable("CryptocurrencySnapshot"))

25/11/24 13:35:31 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

In [10]:
spark.sql("""SELECT * FROM CryptocurrencySnapshot
            LIMIT 10""").show()

+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|Symbol|           Datetime|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|PartitionDate|
+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|   ETH|2025-11-20 15:04:01|     2988.57|     3079.05|       2873.64|         3106.2|             3079.06|   2025-11-20|
|   ETH|2025-11-20 15:04:01|     2988.57|     3079.05|       2873.64|         3106.2|             3079.06|   2025-11-20|
|   ETH|2025-11-20 15:04:20|     2989.23|      3078.3|       2873.64|         3106.2|              3078.3|   2025-11-20|
|   ETH|2025-11-20 15:04:39|      2986.8|     3078.05|       2873.64|         3106.2|             3078.05|   2025-11-20|
|   SOL|2025-11-20 15:04:58|      140.42|       138.1|        130.53|          144.8|               138.1|   2025-11-20|
|   ETH|2025-11-20 15:05:17|    

In [11]:
spark.stop()